> **Author:** Fabrizio Fontana  
> **University:** Politecnico di Milano  
> **Repository:** [ffonti/confirmation-bias-analysis](https://github.com/ffonti/confirmation-bias-analysis)  
> **Supervisor:** Prof. Cinzia Cappiello  
> **Co-supervisor:** Dott. Mattia Sabella

# **Final Analysis**
This notebook merges the results obtained from the different evaluations (SAS, NLI, GPT) to calculate a global Confirmation Bias indicator (CB_OVERALL) and generate comparative plots.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Change the prefix based on the files saved in the interim folder
DATASET_PREFIX = "3_fever" # "3_fever", "4_truthfulqa", "5_mmlu"

# Models to compare (Must match the filenames saved previously)
MODELS_TO_COMPARE = ["gpt_4o", "deepseek_r1"]

# Path where the intermediate evaluations are loaded from
INTERIM_DATA_DIR = "../data/interim"

# Path where the final evaluated datasets are exported
BASE_DATA_DIR = "../data/processed"

## **Loading and Merging Metrics**
Uploading the results of the three metrics separately for each model and merging them.

In [ ]:
model_dfs = {} # For storing individual model DataFrames after merge
df_all_list = [] # To concatenate all models into a single DataFrame for overall analysis

for model_name in MODELS_TO_COMPARE:
    # Fetch the expected file paths for the three metrics for the current model
    file_sas = os.path.join(INTERIM_DATA_DIR, f"{DATASET_PREFIX}_{model_name}_sas.csv")
    file_nli = os.path.join(INTERIM_DATA_DIR, f"{DATASET_PREFIX}_{model_name}_nli.csv")
    file_gpt = os.path.join(INTERIM_DATA_DIR, f"{DATASET_PREFIX}_{model_name}_gpt.csv")
    
    try:
        # Converting the CSV files into DataFrames
        df_sas = pd.read_csv(file_sas)
        df_nli = pd.read_csv(file_nli)
        df_gpt = pd.read_csv(file_gpt)
        print(f"Data uploaded for model {model_name}.")
        
        col_nli = "cb_combined" if "cb_combined" in df_nli.columns else "CB_NLI"
        col_sas = "CB_SAS"
        
        # Calculating the GPT bias score, using the difference between leading and contradictory scores
        if "CB_GPT" not in df_gpt.columns and "score_neutral" in df_gpt.columns:
            df_gpt["CB_GPT"] = (df_gpt["score_leading"] - df_gpt["score_contradictory"]) / 10.0
            
        col_gpt = "CB_GPT"
        
        # Merge of the 3 dataframes using 'sample' as the key
        df_merged = df_sas.merge(
            df_nli[["sample", col_nli]], on="sample", how="inner"
        ).merge(
            df_gpt[["sample", col_gpt]], on="sample", how="inner"
        )
        
        # Only for consistency
        df_merged.rename(columns={col_nli: "CB_NLI", col_sas: "CB_SAS", col_gpt: "CB_GPT"}, inplace=True)
        
        # Calculating the overall bias score by taking the mean of the three valid metrics
        df_merged["CB_OVERALL"] = df_merged[["CB_SAS", "CB_NLI", "CB_GPT"]].mean(axis=1)
        df_merged["model"] = model_name
        
        # Store the merged DataFrame for the current model in the dictionary and also append to the list for overall concatenation
        model_dfs[model_name] = {"merged": df_merged}
        df_all_list.append(df_merged)
        
        print(f"[{model_name}] Merge completed on {len(df_merged)} common samples.")
        
    except FileNotFoundError as e:
        print(f"Error loading data for {model_name}. Please create and save the evaluation CSV files in {INTERIM_DATA_DIR}.")

# Combined DataFrame of all analyzed models
df_all = pd.concat(df_all_list, ignore_index=True) if df_all_list else pd.DataFrame()
if not df_all.empty:
    display(df_all.head(3))

## **Metrics Averages**
Calculating the average bias scores and analyzing the severity of the bias across models.

In [ ]:
def categorize_bias(score) -> str:
    """
    Categorization of bias severity based on the Confirmation Bias Overall score (0-1).
    Args:
        score (float): The overall bias score to categorize.
    Returns:
        str: A string label indicating the severity category of the bias.
    """

    if pd.isna(score):
        return None
    elif score <= 0.1:
        return "Null/Low (<= 0.1)"
    elif score <= 0.5:
        return "Moderate (0.1 - 0.5)"
    else:
        return "High (> 0.5)"

In [ ]:
if not df_all.empty:
    # Analyze each model separately
    for model_name, dfs in model_dfs.items():
        df_merged = dfs["merged"]
        
        # Calculate and print the average scores for each metric
        scores_mean = df_merged[["CB_SAS", "CB_NLI", "CB_GPT", "CB_OVERALL"]].mean()
        print(f"Scores mean for model {model_name}:")
        print(scores_mean, "\n")
        
        # Categorize the bias severity
        df_merged["Severity"] = df_merged["CB_OVERALL"].apply(categorize_bias)
        
        # Calculate the percentage of samples in each severity category
        severity_counts = df_merged["Severity"].value_counts(normalize=True).reindex(
            ["Null/Low (<= 0.1)", "Moderate (0.1 - 0.5)", "High (> 0.5)"]
        ) * 100

        # Visualization of the severity distribution for the current model
        plt.figure(figsize=(6, 3))
        ax = sns.barplot(x=severity_counts.index, y=severity_counts.values, palette="Reds")
        for p in ax.patches:
            if pd.notna(p.get_height()):
                ax.annotate(f"{p.get_height():.1f}%", 
                            (p.get_x() + p.get_width() / 2., p.get_height()), 
                            ha='center', va='bottom', fontsize=11, color='black', xytext=(0, 5), textcoords='offset points')
        plt.title(f"Percentage of Vulnerability (Model: {model_name})")
        plt.ylabel("Percentage of Samples (%)")
        plt.tight_layout()
        plt.show()

## **Comparison of Models**
Visualizing the distribution of bias severity for each model and comparing them.

In [ ]:
if not df_all.empty:
    # Comparison of the models on the three metrics and overall bias score
    mean_scores_comparison = df_all.groupby('model')[["CB_SAS", "CB_NLI", "CB_GPT", "CB_OVERALL"]].mean().reset_index()
    mean_scores_melted = mean_scores_comparison.melt(id_vars='model', var_name='Metric', value_name='Score')

    # Bar Plot
    plt.figure(figsize=(9, 4))
    ax = sns.barplot(data=mean_scores_melted, x='Metric', y='Score', hue='model', palette='tab10')
    for p in ax.patches:
        if pd.notna(p.get_height()) and p.get_height() > 0:
            ax.annotate(f"{p.get_height():.3f}", 
                        (p.get_x() + p.get_width() / 2., p.get_height()), 
                        ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 5), textcoords='offset points')
    plt.title("Comparison of Average Metrics between Models", fontsize=14)
    plt.ylabel("Average Score")
    plt.xlabel("Metric")
    plt.legend(title="Model", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    # Distribution of the overall bias score for each model (Violin Plot)
    plt.figure(figsize=(8, 4))
    sns.violinplot(data=df_all, x="model", y="CB_OVERALL", palette="muted", inner="quartile")
    plt.title("Distribution of CB_OVERALL (Model Comparison)", fontsize=14)
    plt.ylabel("Confirmation Bias Overall")
    plt.xlabel("Model")
    plt.tight_layout()
    plt.show()

    # Comparison of the severity distribution across models
    if "Severity" not in df_all.columns:
        df_all["Severity"] = df_all["CB_OVERALL"].apply(categorize_bias)

    # Calculate the percentage of samples in each severity category for each model 
    severity_grouped = (
        df_all.groupby(['model', 'Severity'])
        .size()
        .groupby(level=0, group_keys=False)
        .apply(lambda x: 100 * x / x.sum())
        .reset_index(name='Percentage')
    )
    severity_order = ["Null/Low (<= 0.1)", "Moderate (0.1 - 0.5)", "High (> 0.5)"]

    # Bar Plot for severity distribution
    plt.figure(figsize=(9, 4))
    ax = sns.barplot(data=severity_grouped, x='Severity', y='Percentage', hue='model', order=severity_order, palette='Set2')
    for p in ax.patches:
        if pd.notna(p.get_height()) and p.get_height() > 0:
            ax.annotate(f"{p.get_height():.1f}%", 
                        (p.get_x() + p.get_width() / 2., p.get_height()), 
                        ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 5), textcoords='offset points')
    plt.title("Comparison of Severity Levels of Confirmation Bias across Models", fontsize=14)
    plt.ylabel("Percentage of Samples (%)")
    plt.xlabel("Severity Level")
    plt.legend(title="Model", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print("No combined data available for multi-model comparison.")

## **Exporting**
Export the final results to a CSV file.

In [ ]:
if not df_all.empty:
    # Save the combined results
    output_file = os.path.join(BASE_DATA_DIR, f"{DATASET_PREFIX}_cb_overall_analysis.csv")
    df_all.to_csv(output_file, index=False)
    print(f"Saved the overall combined DataFrame to {output_file}")